In [ ]:
!pip install BitsandBytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [ ]:
quantisation_config=BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("klyang/MentaLLaMA-chat-7B")
model=AutoModelForCausalLM.from_pretrained("klyang/MentaLLaMA-chat-7B",quantization_config=quantisation_config,device_map="auto",offload_folder="offload")

In [ ]:
def generate_response(conversation_history, user_message):
    """
    Generate a response based on the conversation history and user message.
    """
    conversation_history.append({"role": "user", "content": user_message})
    # Format history for the model
    chat_input = "".join(
        f"{turn['role']}: {turn['content']}\n" for turn in conversation_history
    ) + "assistant:"

    # Tokenize and generate response
    inputs = tokenizer(chat_input, return_tensors="pt", truncation=True).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,  # Randomness
        top_p=0.9,  # Nucleus sampling
        repetition_penalty=1.2  # Penalize repetition
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract assistant's response
    assistant_response = response.split("assistant:")[-1].strip()
    conversation_history.append({"role": "assistant", "content": assistant_response})
    return assistant_response

In [ ]:
conversation_history = []

print("Chatbot: Hello! How can I assist you today?")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break
    response = generate_response(conversation_history, user_input)
    print(f"Chatbot: {response}")